In [ ]:
!pip -q install transformers accelerate sentencepiece fastapi uvicorn nest-asyncio pyngrok

In [ ]:
import torch

from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Loaded!")

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List

app = FastAPI()

class Message(BaseModel):
    role: str
    content: str

class Request(BaseModel):
    question: str
    docs: list
    history: List[Message] = []

In [ ]:
SYSTEM_PROMPT = """
Bạn là chatbot của Trường Đại học Công nghệ - ĐHQGHN (UET).

Bạn CHỈ được sử dụng thông tin trong Context.

QUY TẮC

1. Không sử dụng kiến thức bên ngoài Context.

2. Được phép diễn đạt lại, tóm tắt hoặc tổng hợp thông tin nếu toàn bộ nội dung đều có trong Context.

3. Không được suy đoán, bổ sung hoặc kết luận những điều Context không đề cập.

4. Nếu Context chỉ có một phần thông tin thì chỉ trả lời phần đó.

5. Nếu Context hoàn toàn không có thông tin liên quan thì trả lời đúng nguyên văn:

"Tớ không có đủ dữ liệu để trả lời câu hỏi này."

6. Không được nhắc đến "Context", "Document", "tài liệu", "nguồn", "theo tài liệu",...

7. Luôn trả lời bằng tiếng Việt.

8. Trong hội thoại này:
- "trường"
- "trường mình"
- "UET"
- "Đại học Công nghệ"

đều được hiểu là Trường Đại học Công nghệ - ĐHQGHN.

9. Nếu Context có nhiều thông tin cùng đúng với câu hỏi thì phải liệt kê đầy đủ tất cả.
Không được tự chọn một đáp án nếu Context không khẳng định đó là đáp án duy nhất.

10. Không được suy luận quan hệ sở hữu.
Ví dụ:
- "phục vụ sinh viên UET"
- "dành cho sinh viên UET"
- "sinh viên UET được ở"

không có nghĩa là cơ sở đó thuộc UET.

11. Nếu câu hỏi hỏi về địa điểm, cơ sở, học bổng, chính sách hoặc các lựa chọn mà Context có nhiều đáp án thì phải liệt kê đầy đủ các đáp án trong Context.

12. Chỉ trả lời theo định dạng:

Answer:
<nội dung>
"""


@app.post("/generate")
def generate(req: Request):

    # rẻank lấy top 3
    docs = rerank(
        req.question,
        req.docs,
        top_k=3
    )
    print("=" * 100)
    print("QUESTION:", req.question)
    print("=" * 100)

    for i, doc in enumerate(docs, start=1):
        print("=" * 100)
        print(f"TOP {i}")
        print("Rerank score :", doc["rerank_score"])
        print("Title        :", doc["title"])
        print("Chapter      :", doc.get("chapter"))
        print("Article      :", doc.get("article"))
        print("Heading1     :", doc.get("heading1"))
        print("Heading2     :", doc.get("heading2"))
        print("Heading3     :", doc.get("heading3"))
        print("-" * 100)
        print(doc["text"])
        print()


    print("=" * 80)
    print("AFTER RERANK")

    for i, doc in enumerate(docs, 1):
        print("=" * 80)
        print(f"Top {i}")
        print("Title:", doc["title"])
        print("Chapter:", doc.get("chapter"))
        print("Article:", doc.get("article"))
        print("Score:", doc["rerank_score"])
        print(doc["text"])

    # qwen đọc contexxt
    context = ""

    for i, doc in enumerate(docs, start=1):

        context += f"""
        ### Document {i}

        Title: {doc['title']}

        Chapter: {doc.get('chapter', '')}

        Article: {doc.get('article', '')}

        Heading: {doc.get('heading1', '')}

        Content:
        {doc['text']}

        """

    history_messages = [
        {
            "role": msg.role,
            "content": msg.content
        }
        for msg in req.history
    ]


    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
    ]

    # thêm lịch sử hội thoại
    messages.extend(history_messages)

    # câu hỏi hiện tại
    messages.append(
        {
            "role": "user",
            "content": f"""
    Context:

    {context}

    Question:

    {req.question}
    """
        }
    )

    prompt = tokenizer.apply_chat_template( # chuyển đúng định dạng
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)
    print(context)
    print(prompt)
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    sources = []

    seen = set()

    for doc in docs:
        if doc["title"] not in seen:
            sources.append({
                "title": doc["title"],
                "chapter": doc.get("chapter"),
                "article": doc.get("article")
            })
            seen.add(doc["title"])

    return {
        "answer": answer,
        "sources": sources
    }

In [ ]:
from sentence_transformers import CrossEncoder

print("Loading BGE Reranker...")

reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3"
)

print("Reranker loaded!")


def rerank(query, docs, top_k=5):

    pairs = []

    for doc in docs:

        content = f"""
        Title:
        {doc['title']}

        Chapter:
        {doc.get('chapter','')}

        Article:
        {doc.get('article','')}

        Heading:
        {doc.get('heading1','')}

        Content:
        {doc['text']}
        """

        pairs.append((query, content))

    scores = reranker.predict(
        pairs,
        show_progress_bar=False
    )

    for doc, score in zip(docs, scores):
        doc["rerank_score"] = float(score)

    docs.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return docs[:top_k]

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3Gtam803IhedqXbVcMI6HKjNrGR_7hjHNdFvUm5idNo1Ca932")

In [ ]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()

public_url = ngrok.connect(8000)

print(public_url)

In [ ]:
import uvicorn

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

await server.serve()